# KDIC BGE-M3 Sparse 검색 평가기

BGE-M3의 `lexical_weights`를 이용해 427개 청크를 Sparse 방식으로 검색하고, 검색평가대상 121개 질문의 Top-10 결과를 Gold 청크와 비교합니다.

- 외부 임베딩 API와 API 키는 사용하지 않습니다.
- Colab GPU에서 공식 `BAAI/bge-m3` 모델을 직접 실행합니다.
- Dense 평가와 동일한 업무 필터·Gold·Top-K·8개 지표를 사용합니다.

## 준비 파일

1. `KDIC_BGE_M3_Sparse_평가기.zip`
2. `KDIC_output.zip`
3. `Evaluation_DataSet_v4_1_SearchReady.xlsx`

먼저 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요.

In [ ]:
%pip -q install -U "FlagEmbedding>=1.3.5,<2" "pandas==2.2.2" "openpyxl>=3.1,<4"

import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

WORK_ROOT = Path('/content/kdic_sparse_evaluation')
EVALUATOR_ROOT = WORK_ROOT / 'evaluator'
RESULT_ROOT = WORK_ROOT / 'results'
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
else: print('경고: CPU에서도 실행되지만 매우 느립니다. T4 GPU 런타임을 권장합니다.')

## 1. 파일 업로드

아래 셀을 실행하고 준비한 파일 3개를 한 번에 선택합니다.

In [ ]:
from google.colab import files
uploaded = files.upload()
for filename, content in uploaded.items():
    target = WORK_ROOT / filename
    target.write_bytes(content)
    print(f'업로드: {target.name} ({target.stat().st_size:,} bytes)')

In [ ]:
dataset_candidates = list(WORK_ROOT.glob('*.xlsx'))
kdic_candidates = [p for p in WORK_ROOT.glob('*.zip') if p.name == 'KDIC_output.zip' or p.name.startswith('KDIC_output (')]
evaluator_candidates = [p for p in WORK_ROOT.glob('*.zip') if 'Sparse' in p.name and '평가기' in p.name and '평가결과' not in p.name]
if not dataset_candidates: raise RuntimeError('평가 XLSX를 찾을 수 없습니다.')
if not kdic_candidates: raise RuntimeError('KDIC_output.zip을 찾을 수 없습니다.')
if not evaluator_candidates: raise RuntimeError('KDIC_BGE_M3_Sparse_평가기.zip을 찾을 수 없습니다.')
def choose_latest(candidates): return max(candidates, key=lambda path: path.stat().st_mtime)
DATASET_PATH=choose_latest(dataset_candidates); KDIC_ZIP_PATH=choose_latest(kdic_candidates); EVALUATOR_ZIP_PATH=choose_latest(evaluator_candidates)
if EVALUATOR_ROOT.exists(): shutil.rmtree(EVALUATOR_ROOT)
EVALUATOR_ROOT.mkdir(parents=True)
with zipfile.ZipFile(EVALUATOR_ZIP_PATH) as archive: archive.extractall(EVALUATOR_ROOT)
script_candidates=list(EVALUATOR_ROOT.rglob('evaluate_bge_m3_sparse.py'))
if len(script_candidates)!=1: raise RuntimeError(f'평가 스크립트를 하나로 결정할 수 없습니다: {script_candidates}')
EVALUATOR_SCRIPT=script_candidates[0]
print('평가데이터셋:',DATASET_PATH.name); print('검색 데이터:',KDIC_ZIP_PATH.name); print('평가기:',EVALUATOR_ZIP_PATH.name); print('실행 스크립트:',EVALUATOR_SCRIPT)

## 2. 비교 조건 설정

Dense와 공정하게 비교하려면 아래 값을 고정하세요. T4 메모리가 부족할 때만 `BATCH_SIZE=2`로 낮춥니다.

In [ ]:
BATCH_SIZE=4
MAX_PASSAGE_LENGTH=2048
MAX_QUERY_LENGTH=512
TOP_K=10
print({'검색 방식':'BGE-M3 Sparse lexical matching','모델':'BAAI/bge-m3','batch_size':BATCH_SIZE,'문서 최대 토큰':MAX_PASSAGE_LENGTH,'질문 최대 토큰':MAX_QUERY_LENGTH,'Top-K':TOP_K,'업무 필터':'Gold 업무 사전 필터'})

## 3. Dry-run

모델을 내려받기 전에 질문·청크·Gold 연결을 검사합니다.

In [ ]:
dry_result_dir=WORK_ROOT/'results_dry'
dry_command=[sys.executable,str(EVALUATOR_SCRIPT),'--dataset',str(DATASET_PATH),'--kdic-zip',str(KDIC_ZIP_PATH),'--output-dir',str(dry_result_dir),'--top-k',str(TOP_K),'--max-passage-length',str(MAX_PASSAGE_LENGTH),'--max-query-length',str(MAX_QUERY_LENGTH),'--dry-run']
subprocess.run(dry_command,cwd=EVALUATOR_ROOT,check=True)

## 4. BGE-M3 Sparse 전체 평가

최초 실행 시 약 2.3GB 모델 다운로드와 427개 청크의 Sparse 벡터 생성이 진행됩니다. 이후 같은 런타임에서 다시 실행하면 캐시를 사용합니다.

In [ ]:
RESULT_ROOT.mkdir(parents=True,exist_ok=True)
eval_command=[sys.executable,str(EVALUATOR_SCRIPT),'--dataset',str(DATASET_PATH),'--kdic-zip',str(KDIC_ZIP_PATH),'--output-dir',str(RESULT_ROOT),'--top-k',str(TOP_K),'--batch-size',str(BATCH_SIZE),'--max-passage-length',str(MAX_PASSAGE_LENGTH),'--max-query-length',str(MAX_QUERY_LENGTH)]
subprocess.run(eval_command,cwd=EVALUATOR_ROOT,check=True)

## 5. 평가 결과 확인

In [ ]:
summary=json.loads((RESULT_ROOT/'summary.json').read_text(encoding='utf-8'))
overall=pd.DataFrame([summary['overall']]); by_domain=pd.read_csv(RESULT_ROOT/'summary_by_domain.csv'); questions=pd.read_csv(RESULT_ROOT/'question_results.csv')
print('검색 방식:',summary['retriever']); print('전체 평가 결과'); display(overall); print('도메인별 평가 결과'); display(by_domain); print('Hit@3 실패 질문 예시')
display(questions.loc[questions['hit_at_3']==0,['evaluation_id','question','domain','gold_chunk_ids','retrieved_chunk_ids']].head(10))

## 6. 결과 다운로드

다운로드한 ZIP을 `KDIC 검색평가 비교 대시보드`에 넣으면 Dense 결과와 함께 비교할 수 있습니다.

In [ ]:
from google.colab import files
archive_path=shutil.make_archive('/content/KDIC_BGE_M3_Sparse_평가결과','zip',RESULT_ROOT)
print('결과 압축:',archive_path)
files.download(archive_path)